# Raw-Feature Baseline (no topomaps)

Tests whether feeding the raw 260 features directly into the GRU
beats the topomap+CNN approach (honest masked RMSE: 0.301).

Same masking, same metric, same loop feature, same train/val split.
Only the representation changes: raw features instead of CNN-encoded topomaps.

In [14]:
import os, json, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler
from pathlib import Path
from glob import glob

TRAIN_DIR = Path('train')
VAL_DIR = Path('val')

# EEG feature columns (256) + phys (4) = 260 features; arousal is the target
def load_folder(folder):
    rows = []
    for f in sorted(glob(str(folder / '*.csv'))):
        df = pd.read_csv(f).dropna().reset_index(drop=True)
        if len(df) < 4:
            continue
        stem = Path(f).stem.replace('Features_', '')
        pid, tid = stem.split('-')
        # loop assignment, same logic as topomap pipeline
        total = len(df) - (len(df) % 4)
        df = df.iloc[:total].copy()
        bins_per_loop = total // 4
        df['participant_id'] = pid
        df['treatment_id'] = tid
        df['loop_num'] = (np.arange(total) // bins_per_loop).clip(max=3) + 1
        df['bin_within_loop'] = np.arange(total) % bins_per_loop
        df['filename'] = f"{pid}_{tid}"
        rows.append(df)
    return pd.concat(rows, ignore_index=True)

print("Loading train...")
train_df = load_folder(TRAIN_DIR)
print("Loading val...")
val_df = load_folder(VAL_DIR)

feature_cols = [c for c in train_df.columns if c.startswith('EEG_')] + \
               ['heartrate_mean', 'GSR_mean', 'IRPleth_mean', 'Respir_mean']
print(f"Feature count: {len(feature_cols)}")   # expect 260
print(f"Train rows: {len(train_df)}, Val rows: {len(val_df)}")

Loading train...
Loading val...
Feature count: 260
Train rows: 245100, Val rows: 83520


In [15]:
# Fit feature scaler on ALL 260 features (train only)
scaler_X = StandardScaler()
scaler_X.fit(train_df[feature_cols].values)

# Fit arousal target scaler (train only)
scaler_y = StandardScaler()
scaler_y.fit(train_df[['LABEL_SR_Arousal']].values)

print("Scalers fit on train.")
print(f"Feature means sample: {scaler_X.mean_[:3]}")
print(f"Arousal mean/std: {scaler_y.mean_[0]:.4f} / {scaler_y.scale_[0]:.4f}")

Scalers fit on train.
Feature means sample: [5.59554559 4.11847716 6.25564936]
Arousal mean/std: 0.4858 / 0.3109


In [16]:
MAX_LEN = 38

class RawFeatureDataset(Dataset):
    def __init__(self, df, scaler_X, scaler_y, feature_cols):
        self.scaler_X = scaler_X
        self.scaler_y = scaler_y
        self.feature_cols = feature_cols
        self.samples = []
        for (fname, loop), g in df.groupby(['filename', 'loop_num']):
            g = g.sort_values('bin_within_loop')
            self.samples.append({
                'features': g[feature_cols].values.astype(np.float32),
                'loop_num': loop,
                'targets': g['LABEL_SR_Arousal'].values.astype(np.float32),
            })
        print(f"Dataset: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        feats = self.scaler_X.transform(s['features'])             # (real_len, 260)
        targets = self.scaler_y.transform(
            s['targets'].reshape(-1, 1)).flatten()                 # (real_len,)

        real_len = feats.shape[0]
        if real_len < MAX_LEN:
            pad = MAX_LEN - real_len
            feats = np.pad(feats, ((0, pad), (0, 0)))
            targets = np.pad(targets, (0, pad))
        elif real_len > MAX_LEN:
            feats = feats[:MAX_LEN]; targets = targets[:MAX_LEN]; real_len = MAX_LEN

        mask = np.zeros(MAX_LEN, dtype=np.float32)
        mask[:real_len] = 1.0
        loop_norm = (s['loop_num'] - 1) / 3.0

        return (torch.FloatTensor(feats),
                torch.FloatTensor([loop_norm]),
                torch.FloatTensor(targets),
                torch.FloatTensor(mask))

In [17]:
train_ds = RawFeatureDataset(train_df, scaler_X, scaler_y, feature_cols)
val_ds   = RawFeatureDataset(val_df, scaler_X, scaler_y, feature_cols)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

# verify one batch
feats, loop, target, mask = next(iter(train_loader))
print("feats:", feats.shape)      # (32, 38, 260)
print("loop:", loop.shape)        # (32, 1)
print("target:", target.shape)    # (32, 38)
print("mask:", mask.shape)        # (32, 38)

Dataset: 8168 samples
Dataset: 2784 samples
feats: torch.Size([32, 38, 260])
loop: torch.Size([32, 1])
target: torch.Size([32, 38])
mask: torch.Size([32, 38])


In [18]:
class RawGRUModel(nn.Module):
    def __init__(self, n_features=260, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()
        gru_input = n_features + 1  # +1 for loop number
        self.gru = nn.GRU(gru_input, hidden_dim, num_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 1)
        )

    def forward(self, feats, loop_num):
        seq_len = feats.shape[1]
        loop_exp = loop_num.unsqueeze(1).expand(-1, seq_len, -1)  # (B, seq, 1)
        x = torch.cat([feats, loop_exp], dim=-1)                  # (B, seq, 261)
        out, _ = self.gru(x)
        return self.head(out).squeeze(-1)                         # (B, seq)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RawGRUModel().to(device)
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
Parameters: 613,249


In [19]:
import time

EXPERIMENT = "raw_features_gru"
DESCRIPTION = "Raw 260 features -> GRU, masked loss/RMSE, loop feature, no topomap"
os.makedirs("models", exist_ok=True)
BEST_MODEL_PATH = f"models/{EXPERIMENT}_best.pt"
RESULTS_LOG = "models/results_log.jsonl"

EPOCHS = 50
LEARNING_RATE = 1e-3

optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

def train_one_epoch():
    model.train()
    total = 0
    for feats, loop, target, mask in train_loader:
        feats, loop = feats.to(device), loop.to(device)
        target, mask = target.to(device), mask.to(device)
        optimizer.zero_grad()
        out = model(feats, loop)
        loss = (((out - target) ** 2) * mask).sum() / mask.sum()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(train_loader)

def evaluate():
    model.eval()
    total = 0
    preds, targs = [], []
    with torch.no_grad():
        for feats, loop, target, mask in val_loader:
            feats, loop = feats.to(device), loop.to(device)
            target, mask = target.to(device), mask.to(device)
            out = model(feats, loop)
            loss = (((out - target) ** 2) * mask).sum() / mask.sum()
            total += loss.item()
            mb = mask.bool()
            preds.append(out[mb].cpu().numpy())
            targs.append(target[mb].cpu().numpy())
    preds = np.concatenate(preds)
    targs = np.concatenate(targs)
    preds = scaler_y.inverse_transform(preds.reshape(-1, 1)).flatten()
    targs = scaler_y.inverse_transform(targs.reshape(-1, 1)).flatten()
    rmse = math.sqrt(np.mean((preds - targs) ** 2))
    return total / len(val_loader), rmse

In [20]:
best_val_rmse = float("inf")
best_epoch = 0

print(f"{'Epoch':>6} {'Train':>10} {'ValLoss':>10} {'ValRMSE':>10} {'LR':>10}")
print("-" * 50)
for epoch in range(1, EPOCHS + 1):
    tr = train_one_epoch()
    vl, vr = evaluate()
    scheduler.step(vl)
    if vr < best_val_rmse:
        best_val_rmse, best_epoch = vr, epoch
        torch.save({"experiment": EXPERIMENT, "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "val_rmse": vr}, BEST_MODEL_PATH)
        marker = " *"
    else:
        marker = ""
    lr = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} {tr:>10.4f} {vl:>10.4f} {vr:>10.4f} {lr:>10.1e}{marker}")

print(f"\nBest Val RMSE: {best_val_rmse:.6f} at epoch {best_epoch}")

with open(RESULTS_LOG, "a") as f:
    f.write(json.dumps({"experiment": EXPERIMENT, "description": DESCRIPTION,
                        "best_val_rmse": round(best_val_rmse, 6),
                        "best_epoch": best_epoch,
                        "timestamp": time.strftime("%Y-%m-%d %H:%M")}) + "\n")

 Epoch      Train    ValLoss    ValRMSE         LR
--------------------------------------------------
     1     0.9164     1.2059     0.3414    1.0e-03 *
     2     0.8021     1.5926     0.3924    1.0e-03
     3     0.7370     1.1613     0.3351    1.0e-03 *
     4     0.7007     1.1948     0.3398    1.0e-03
     5     0.6778     1.2663     0.3498    1.0e-03
     6     0.6716     1.2910     0.3532    1.0e-03
     7     0.6550     1.1887     0.3390    1.0e-03
     8     0.6468     1.2229     0.3438    1.0e-03
     9     0.6402     1.2949     0.3538    5.0e-04
    10     0.6017     1.1999     0.3405    5.0e-04
    11     0.5928     1.1997     0.3405    5.0e-04
    12     0.5819     1.1611     0.3349    5.0e-04 *
    13     0.5799     1.3424     0.3602    5.0e-04
    14     0.5732     1.2989     0.3543    5.0e-04
    15     0.5692     1.2186     0.3432    5.0e-04
    16     0.5654     1.1280     0.3302    5.0e-04 *
    17     0.5616     1.1053     0.3269    5.0e-04 *
    18     0.5591    

In [21]:
import pandas as pd
import numpy as np

f = 'train/Features_P057-T15.csv'   # one of the flagged files
d = pd.read_csv(f)
nan_rows = d[d.isna().any(axis=1)]
print(f"{len(nan_rows)} NaN rows in {f}")

# For each NaN row, which columns are missing?
for i, row in nan_rows.iterrows():
    missing = row.index[row.isna()].tolist()
    eeg_missing = [c for c in missing if c.startswith('EEG_')]
    other_missing = [c for c in missing if not c.startswith('EEG_')]
    print(f"  row {i}: {len(eeg_missing)} EEG cols missing, other missing: {other_missing}")

1 NaN rows in train/Features_P057-T15.csv
  row 0: 0 EEG cols missing, other missing: ['LABEL_SR_Arousal']
